In [2]:
import ee
import geemap
from datetime import datetime
import os
import geopandas as gpd

ee.Authenticate()

m = geemap.Map(center=(46.2, 7.657715), zoom=9) #Valais
m


Successfully saved authorization token.


Map(center=[46.2, 7.657715], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [3]:
# now load shapefile of the canton of Valais and add it to the map
geom = geemap.shp_to_ee('valais_canton.shp')
m.addLayer(geom, {}, 'Valais')

In [4]:
# also try to load all the parcels
parcels = geemap.shp_to_ee('parcels_select.shp')
m.addLayer(parcels, {}, 'Parcels')

In [5]:
# -----------------------------
# 2. Define time range and other parameters
# -----------------------------
start_date = '2023-03-01'
end_date   = '2023-10-31'
resol1 = 10
resol2 = 20
cloud_thresh = 20


In [6]:
s2Swiss = ee.ImageCollection("projects/satromo-prod/assets/col/S2_SR_HARMONIZED_SWISS") \
    .filterDate(start_date, end_date) \
    .filterBounds(geom) \
    .filterMetadata('SENSING_ORBIT_NUMBER', 'equals', 108)

s2_filtered_10 = s2Swiss.filter(ee.Filter.stringContains("system:index", f"bands-{resol1}m"))
s2_filtered_20 = s2Swiss.filter(ee.Filter.stringContains("system:index", f"bands-{resol2}m"))

image_list10 = s2_filtered_10.aggregate_array('system:index').getInfo()
image_list20 = s2_filtered_20.aggregate_array('system:index').getInfo()
total_imgs = len(image_list10) + len(image_list20)
print(f"Nombre d'images à analyser : {total_imgs}")

results = []

Nombre d'images à analyser : 60


In [7]:
# because the 20m and 10m come in seperate images, we first need to link them into one image per date, based on the date in the properties
s2_bands = s2_filtered_20.first().bandNames()# get the band names of the 20m image, which are the same as the 10m image, just with different resolution in the name

# Link cloud score to image.
s2_total = s2_filtered_10.linkCollection(s2_filtered_20, s2_bands)
s2_total

In [9]:
clip_coll = (
    s2_total.map(lambda image: image.clip(geom))
)
clip_coll

In [10]:
# show the ninth image in the collection (seems that only orbit 108 covers the whole of Wallis?)
visparams = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 4000,
    'gamma': 1.0
}

xth_image = clip_coll.toList(clip_coll.size()).get(7)
m.addLayer(ee.Image(xth_image), visparams, 'Xth image in collection')
m

Map(bottom=46825.0, center=[46.20074541128312, 7.657470703125001], controls=(WidgetControl(options=['position'…

In [ ]:
# show the value of the cloudProbability band of the xth-image in the collection
check_image = ee.Image(xth_image)
cloudProb_value = check_image.select('cloudProbability').reduceRegion(reducer=ee.Reducer.median(), geometry=geom, scale=10, bestEffort=True).values().get(0).getInfo()
print(f"cloudProb-value for the xth image: {cloudProb_value}")

cloudProb-value for the xth image: 7.000000000000005


In [12]:
# map the cloudProb value to the image collection
s2_filtered_cs_geom = clip_coll.map(lambda img: img.set('cloudProbability', img.select('cloudProbability').reduceRegion(reducer=ee.Reducer.median(), geometry=geom, scale=10, bestEffort=True).values().get(0)))


In [13]:
# select only images with cloudProb < cloud_thresh
s2_filtered_cs_thresh = s2_filtered_cs_geom.filter(ee.Filter.lt('cloudProbability', cloud_thresh))
print(f"Nombre d'images après filtrage sur cloudProb < {cloud_thresh} : {s2_filtered_cs_thresh.size().getInfo()}")


Nombre d'images après filtrage sur cloudProb < 20 : 19


In [ ]:
## we need to make sure that non of the parcels is covered by snow during the analysis period. As a matter of fact, we can also do the cloud masking just based on the parcels?
# import geopandas as gpd
# gdf = gpd.read_file('parcels_shapefile.shp', encoding='latin1')
# parcels = geemap.gdf_to_ee(gdf)
# m.addLayer(parcels, {}, 'Parcels')

In [ ]:
# # compute NDSI for the xth image
# ndsi_image = check_image.normalizedDifference(['B3', 'B11']).rename('NDSI')
# ndsi_value = ndsi_image.reduceRegion(reducer=ee.Reducer.median(), geometry=geom, scale=10, bestEffort=True).values().get(0).getInfo()
# print(f"NDSI value for the xth image: {ndsi_value}")

In [ ]:
# next step would be to use the parcels as geometry for the cloud masking, or at least for snow masking. 
# Don't have to use it as a strict mask, but parcels that are covered by snow should be somehow excluded to not be misclassified as moqing event.
# Use NDSI. 

# how come the bands are not all in there? deepending on resolution, some bands are missing. For example, B11 is not in the 10m resolution images, but only in the 20m ones. So we need to make sure to use the right bands for the NDSI calculation depending on the resolution of the image.We want all
